# Data Generator

Here we  generate the synthetic datasets used for neutronstar EoS inference by constructing candidate equations of state and solving the Tolman–Oppenheimer–Volkoff(TOV) equations.

The neutron star EoS consists into three regions:

- Low-density region: described by the tabulated SLy or AP4 EoS.
- High-density region: extended using a sampled speed-of-sound parametrization defined by interpolation nodes $(\rho_i,c_{s,i})$.
- High-pressure core: modified by a vacuum-energy shift $\Lambda$ associated with the phase transition.

The data-generation pipeline is

$$(\rho_i,c_{s,i}),\ \Lambda,\ \mathrm{SLy/AP4} \;\longrightarrow\; \mathrm{EoS} \;\longrightarrow\; \mathrm{TOV\ solver} \;\longrightarrow\; (M,R,k_2) \;\longrightarrow\; \mathrm{noise\ injection} \;\longrightarrow\; \mathrm{ML\ dataset}$$

For each EoS realization, the TOV equations are solved over a range of pressures to construct the corresponding stellar sequence. Physically accepted models are then retained, after which noisy synthetic observations are sampled for use in the inference pipeline.

This notebook generates:
1. training datasets,
2. testing datasets,
3. numerical TOV solutions used for physical validation.

We first define the necessary libraries, physical constants needed for the construction of our data.

In [2]:
from joblib import Parallel, delayed
from scipy.interpolate import interp1d
from scipy.stats import norm, uniform
import matplotlib.pyplot as plt
from scipy.optimize import root
from numba import njit
from PIL import Image
import pandas as pd
import numpy as np
import sympy as sp
import statistics
import random
import time
import os

In [3]:
#constants
c_light = 2.997*1e10 #cm s^-1
hbar = 6.582*1e-22 #MeV s 
MeV = 1.602*1e-6 #g cm^2 s^-1
Kp = MeV / (hbar**3 * c_light**3) #in g and cm
Krho = MeV/(hbar**3 * c_light**5) #in g and cm
fm = (1e-13)**(-3) #fm^-3 in cm^-3
n0 = 0.16 * fm #nuclear saturation density in cm^-3
m = 1.675 * 1e-24 #neutron mass in g
M_sun = 1.988 * 1e33 #sun mass in g
G_const = 6.67*1e-11*1e6/1e3 ##cm^3/g/s^2
Lu = (1 / M_sun) * (c_light ** 2) * (1/G_const) #1cm in G=c=Msun=1
Pu = (1/M_sun) / (Lu ** 3) / (c_light **2) 
rhou = (1/ M_sun ) / (Lu **3)
KK = ((2.997*1e5)**2)/(6.67*1e-20 * 1.988 *1e30)

In [4]:
rho_t = 2*n0*m*rhou
rho_fin = 12*n0*m*rhou
pc = 200**4 * Kp * Pu
r0 = 1e-5
a0=1
f0=1
h0=1
H0 = a0 * (r0 ** 2)
beta0 = 2 * a0 * r0
rspan = (r0, 200)
Lamb_arr = [-(194.0**4) * Kp * Pu, -(150.0**4) * Kp * Pu, -(120.0**4) * Kp * Pu, -(95.0**4) * Kp * Pu, -(50.0**4) * Kp * Pu,
            0, (50.0**4) * Kp * Pu, (95.0**4) * Kp * Pu, (120.0**4) * Kp * Pu, (194.0**4) * Kp * Pu]
M_norm = 3
R_norm = 20

In [19]:
ap4 = pd.read_csv("data_reference/Rescaledap4.dat", sep = r'\s+', header=None)
sly = pd.read_csv("data_reference/Rescaledsly.dat", sep = r'\s+', header=None)
ap4.columns=['p', 'eps', 'rho']
sly.columns=['p', 'eps', 'rho']

#training data
rho_train = pd.read_csv("data_reference/matrixrho2.dat", sep = r'\s+', header=None)
cs_train = pd.read_csv("data_reference/matrixcs2.dat", sep = r'\s+', header=None)
rho_train.columns=['col1', 'col2', 'col3', 'col4', 'col5', 'col6', 'col7']
cs_train.columns=['col1', 'col2', 'col3', 'col4', 'col5', 'col6', 'col7']

#test data
rho_test = pd.read_csv("data_reference/matrixrhoTest.dat", sep = r'\s+', header=None)
cs_test = pd.read_csv("data_reference/matrixcsTest.dat", sep = r'\s+', header=None)
rho_test.columns=['col1', 'col2', 'col3', 'col4', 'col5', 'col6', 'col7']
cs_test.columns=['col1', 'col2', 'col3', 'col4', 'col5', 'col6', 'col7']

sly = sly.values
ap4 = ap4.values

rho_train = rho_train.values
cs_train = cs_train.values
rho_test = rho_test.values
cs_test = cs_test.values

### High-density EoS extension

The inital EoS is extended into the high-density regime using the sampled speed-of-sound parametrization. Firstly, the transition point in the original EoS is located and the local derivative $dp/d\varepsilon$ is calculated. Then, we linearly interpolate the nodes $(\rho_i,c_{s,i})$.

Using $$c_s^2 = \frac{dp}{d\varepsilon},$$ the pressure and energy density are evolved toward higher mass densities. The generated high-density segment is finally then merged with the original low-density EoS to obtain the final complete EoS table used by the TOV solver.

In [6]:
def find_pos(matrix, val, col):
  """
  Find element index in a matrix column closest to a specific value.

  Parameters:
      matrix: ndarray
          2D array with data
      val: float
          Value to find in column.
      col: int
          Column index of matrix where search is done.

  Returns:
      int
          Row index of element which value in column is closest to target.
  """
  col_data = matrix[:, col]
  for i in range(len(col_data) - 1):
    dx1 = abs(col_data[i] - val)
    dx2 = abs(col_data[i+1] - val)

    if col_data[i+1] >= val:
      if dx1<dx2:
        return i
      else:
        return i+1

def deriv(eos, max_idx):
  """Compute the derivative of the EOS.

  Parameters:
      eos: ndarray
          2D array with EOS data, with columns being parameters.
      max_i: int
          Index for which we compute derivative up to.

  Returns:
      dp / deps: float
          Derivative of pressure with respect to the energy density.
  """

  dp = eos[max_idx + 1, 0] - eos[max_idx, 0]
  deps = eos[max_idx + 1, 1] - eos[max_idx, 1]

  return dp / deps

def cs_interpolate(eos, max_i, p1, matrix_rho, matrix_cs, ds):
  """
  Create linear interpolation for speed of sound c_s for 7 points.

  Parameters:
      eos: ndarray
          Equation of state.
      max_i: int
          Index for which EOS interpolation starts.
      p1: float
          pressure at first interpolation point
      matrix_rho: ndarray
          Matrix with density data.
      matrix_cs: ndarray
          Matrix with speed of sound data.
      ds: int
          Row index which selects interval of rho and speed of sound values.

  Returns
      cs_interpolation
          Function which interpolates the speed of sound given datasets.
  """
  segment_mat = []
  segment_mat.append([eos[max_i, 2], np.sqrt(p1)])

  for i in range(7):
    segment_mat.append([matrix_rho[ds, i], matrix_cs[ds, i]])

  segment_mat = np.array(segment_mat)
  cs_interpolation = interp1d(segment_mat[:,0], segment_mat[:,1], kind='linear', fill_value='extrapolate')

  return cs_interpolation

def EOS_HE(he_eos, eos, max_i, cs_interpolation, rho_final):
  """
  Extend the EOS table to higher densities by interpolation speed of sound. Starting from the
  maximum index, increase the density and update energy density and pressure, using the
  speed of sound interpolations. These new rows are appended to the dataset until we arrive
  at the final density node.

  Parameters
      he_eos: ndarray
          EOS array which we will increase with data.
      eos: ndarray
          EOS table which initial data are obtained from.
      max_i: int
          Index from which we start in the EOS row.
      cs_interpolation: function
          Function returning speed of sound with respect to density.
      rho_final: float
          Final density up to which we determine EOS.

  Returns
      he_eos: ndarray
          EOS array which is enlargened.
  """
  drho = 1e-5

  M = int((1e-3 - eos[max_i, 2])/1e-5) + int((1e-2 - 1e-3)/1e-4) + int((rho_final - 1e-2)/1e-3)

  p_last, eps_last, rho_last = eos[max_i,0], eos[max_i,1], eos[max_i,2]

  for i in range(M-1):
    p_next = p_last + cs_interpolation(rho_last)**2 * drho * (eps_last + p_last)/rho_last
    eps_next = eps_last + drho * (eps_last + p_last) / rho_last
    rho_next = rho_last + drho

    if rho_next > rho_final:
      break

    #append row
    he_eos = np.vstack([he_eos, [p_next, eps_next, rho_next]])

    p_last, eps_last, rho_last = p_next, eps_next, rho_next

    if rho_next < 1e-3:
      drho = 1e-5
    elif rho_next < 1e-2:
      drho = 1e-4
    elif rho_next < 1e-1:
      drho = 1e-3
    else:
      drho = 1e-2

  return he_eos

def mergeEOS(eos_mat, he_eos, eos, max_idx):
  """
  Add new data to original EOS matrix.

  Parameters
      eos_mat: ndarray
          Array which stores the total EOS.
      he_eos: ndarray
          Extension of EOS with new rows.
      eos: ndarray
          Initial EOS data table.
      max_i: int
          Index up to which we include rows from EOS

  Returns
      eos_mat: ndarray
          Merged EOS array.
  """
  eos_mat = np.vstack([eos_mat, eos[:max_idx+1,:]])
  eos_mat = np.vstack([eos_mat,he_eos])
  return eos_mat

### Constructing the complete EoS

The following function combines the previous steps into a single pipeline. For a chosen transition density $\rho_{treshold}$, it locates the corresponding point in the initial EoS and constructs the sampled high-density $c_s(\rho)$ interpolation. Then, it extends the EoS up to $\rho_{final}$, and merges both density regimes into a complete EoS table.

In [7]:
def build(eos, matrix_rho, matrix_cs, ds, rho_treshold, rho_final):
  """
  Create an EOS table by including higher densities to our original model. Here we find the
  transition point, apply a speed of sound interpolation and generate the EOS extension.
  Subsequently, we merge this with our original EOS data

  Parameters
      eos: ndarray
          baseline EOS data.
      matrix_rho: ndarray
          Matrix with base density values used for interpolation.
      matrix_cs: ndarray
          Matrix with speed of sound values used for interpolation.
      ds: int
          Row index for which chooses which interpolation interval to use.
      rho_treshold: float
          Density value for which we start extending EOS.
      rho_final: float
          Density up to which EOS is extended.

  Returns
      eos_matrix: ndarray
          Final EOS table including extended data.
  """
  i = find_pos(eos, rho_treshold, 2)
  p1 = deriv(eos, i)
  cs_interpolation = cs_interpolate(eos, i, p1, matrix_rho, matrix_cs, ds)

  he_eos = np.empty((0,3))
  he_eos = EOS_HE(he_eos, eos, i, cs_interpolation, rho_final)

  eos_matrix = np.empty((0,3))
  eos_matrix = mergeEOS(eos_matrix, he_eos, eos, i)

  return eos_matrix

### EoS interpolation for TOV integration

During the TOV integration, the energy density $\varepsilon$ and its derivative $d\varepsilon/dp$ have to be evaluated continuously as functions of pressure. Since the EoS is stored discretely on a pressure grid, we use piecewise linear interpolation for $\varepsilon(p)$ and the corresponding derivatives for $d\varepsilon/dp$.

Above the transition pressure $p_c$, the vacuum-energy shift $\Lambda$ is included by shifting the pressure argument and energy density by it. These quantities are then used in the TOV equations.

In [28]:
@njit
def linear_interp(x, p, eps):
    """
    Perform linear interpolation to determine the value corresponding to a given point.
    We locate the interval containing the input and linearly interpolate between
    the two surrounding data points. Values outside the provided range are extrapolated
    using the nearest interval.

    Parameters
        x: float
            Input value at which we interpolate.
        p: ndarray
            Array containing the interpolation points.
        eps: ndarray
            Array containing the values corresponding to the interpolation points.

    Returns
        eps_interp: float
            interpolated value (being x).
    """
    idx = np.searchsorted(p, x) - 1

    if idx < 0:
        idx = 0
    elif idx >= len(p) - 1:
        idx = len(p) - 2

    p0 = p[idx]
    p1 = p[idx + 1]
    e0 = eps[idx]
    e1 = eps[idx + 1]

    return e0 + (e1 - e0) * (x - p0) / (p1 - p0)

@njit
def eps_prime_interp(x, p, slopes):
    """
    Find the derivative of a piecewise linear interpolation at a given point.
    We locate the interval containing the input and return the slope associated 
    with the interval. Values outside our range use the slope of the nearest interval.

    Parameters
        x: float
            Input value at the derivative is determined.
        p: ndarray
            Array containing the interpolation points that define the intervals.
        slopes: ndarray
            Array containing the constant slope for each interpolation interval.

    Returns
        eps_prime: float
            Slope of the piecewise linear interpolation at x.

    """
    idx = np.searchsorted(p, x, side="right") - 1

    if idx < 0:
        idx = 0
    elif idx >= len(slopes):
        idx = len(slopes) - 1

    return slopes[idx]

@njit
def eos_values(x, p, eps, slopes, Lambda):
    """
    Evaluate the EOS value and its derivative at a pressure. Below the
    transition pressure pc, we use the regular linear interpolation. Above the 
    transition pressure pc, we shift the interpolation argument and
    energy density by Lambda.
 
    Parameters
        x: float
            Pressure value at which EOS is evaluated.
        p: ndarray
            Array containing the pressure interpolation points.
        eps: ndarray
            Array containing the energy density values corresponding to p.
        slopes: ndarray
            Array containing the constant slope for each interpolation interval.
        Lambda: float
            Shift applied to the pressure and energy density above the
            transition pressure.

    Returns
        eps_value: float
            Interpolated energy density corresponding to x.
        deps_value: float
            Derivative of the interpolated energy density with respect to pressure.
    """
    if x < pc:
        eps_value = linear_interp(x, p, eps)
        deps_value = eps_prime_interp(x, p, slopes)
    else:
        eps_value = linear_interp(x + Lambda, p, eps) + Lambda
        deps_value = eps_prime_interp(x + Lambda, p, slopes)

    return eps_value, deps_value

### TOV and tidal-perturbation equations

The neutron-star structure is obtained by integrating the TOV system together with the tidal perturbation equations. At each radius, the current pressure is used to evaluate the EoS quantities $\varepsilon(P)$ and $d\varepsilon/dP$, which then determine the radial evolution of the metric functions, pressure, and tidal variables.

The state vector is $$ u(r) = \left[f(r),\,h(r),\,P(r),\,H(r),\,\beta(r)\right],$$ and this function returns the corresponding derivatives with respect to radius. These derivatives are then integrated numerically to obtain the stellar structure and tidal response.

In [9]:
@njit
def tov_equations(r, u, p, eps_table, slopes, Lambda):
  """
    Compute the Tolman-Oppenheimer-Volkoff (TOV) equations and tidal perturbation
    equations. Here we evaluate the EOS at the current pressure and calculate the
    derivatives of the metric functions, pressure, tidal field and its derivative.

    Parameters
        r: float
            Radial coordinate.
        u: ndarray
            Array containing the variables [f, h, P, H, beta].
        p: ndarray
            Array containing the pressure interpolation points of the EOS.
        eps_table: ndarray
            Array containing the energy density values corresponding to p.
        slopes: ndarray
            Array containing the derivative of the energy density in each
            interpolation interval.
        Lambda: float
            Shift parameter applied to the EOS above the transition pressure.

    Returns
        ndarray
            Array containing the radial derivatives [df/dr, dh/dr, dP/dr, dH/dr, dbeta/dr].
    """
  f, h, P, H, beta = u

  eps, deps = eos_values(P, p, eps_table, slopes, Lambda)

  du1 = (1 - f - 8*np.pi * (r ** 2) * eps) / r
  du2 = -(h * (-1 + f - 8 * np.pi * (r ** 2) * P)) / (r * f)
  du3 = ((-1 + f - 8 * np.pi * (r**2)*P) * (P+eps)) / (2*r*f)
  du4 = beta
  du5 = (H * (- (f**3)
              + (1 + 8 * np.pi * (r**2) * P)**3
              - f * (1 + 8 * np.pi * (r**2) * P) * (-3+60*np.pi * (r**2)*P + 20 * np.pi * (r**2) * eps)
              + (f**2) * (-3
                          + 60 * np.pi * (r**2) * P
                          + 8 * np.pi * (r**3) * deps * (-1 + f - 8 * np.pi * (r**2)*P) * (P+eps)/(2 * r * f)
                          + 20 * np.pi * (r**2) * eps
          )) + r * f * (-1 + f - 8 * np.pi * (r**2) * P) * (1 + f + 4 * np.pi * (r**2) * P - 4 * np.pi * (r**2) * eps) * beta
  ) / ((r**2) * (f**2) * (1 - f + 8 * np.pi * (r**2) * P))

  return np.array([du1, du2, du3, du4, du5])

### Numerical integration

The coupled TOV and tidal equations are integrated radially outward from the stellar center using a fourth order Runge–Kutta (RK4) scheme with a fixed step size. For a given central pressure $P_0$, the state vector is evolved as $$\frac{du}{dr} = F(r,u;\mathrm{EoS}),$$ until the pressure becomes negligible, such that $$\frac{P(r)}{P_0} < 10^{-12},$$ which defines the numerical stellar surface. The final radius and state variables are then used to calculate our global stellar observables.

In [10]:
@njit
def integrator(P0, p, eps, slopes, Lambda):
    """
    Integrate the TOV equations using the fourth order Runge-Kutta method with a fixed 
    radial step size. Starting from the central pressure and initial stellar conditions, 
    we evolve the system outward until the pressure becomes negligible or the maximum radius is obtained.

    Parameters
        P0: float
            Central pressure used as the initial pressure of the star.
        p: ndarray
            Array containing the pressure interpolation points of the EOS.
        eps: ndarray
            Array containing the energy density values corresponding to p.
        slopes: ndarray
            Array containing the derivative of the energy density in each
            interpolation interval.
        Lambda: float
            Shift parameter applied to the EOS above the transition pressure.

    Returns
        r: float
            Final radial coordinate reached by the integration.
        u: ndarray
            Array containing the final values [f, h, P, H, beta].
    """
    dt = 0.0005
    r = rspan[0]
    u = np.array([f0, h0, P0, H0, beta0], dtype=np.float64)

    while r < rspan[1]:
        k1 = tov_equations(r, u, p, eps, slopes, Lambda)
        k2 = tov_equations(r + dt / 2, u + dt * k1 / 2, p, eps, slopes, Lambda)
        k3 = tov_equations(r + dt / 2, u + dt * k2 / 2, p, eps, slopes, Lambda)
        k4 = tov_equations(r + dt, u + dt * k3, p, eps, slopes, Lambda)

        u = u + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)
        r += dt

        if u[2] / P0 < 1e-12:
            break

    return r, u

### Generating stellar sequences

For a fixed EoS, neutron-star configurations are made by solving the TOV equations over a range of central pressures $P_0$. Each central pressure corresponds to a different stellar model.

After each integration, the final state variables are used to compute the stellar mass, radius, compactness, Tidal Love number $k_2$, and dimensionless tidal deformability. These quantities are stored to construct the mass–radius and tidal-response relations associated with the chosen EoS.

The central pressure spacing is increased at larger pressures to reduce the amount of integrations while still sampling the stellar sequence across the full pressure range.

In [10]:
def cycle_tov(data_matrix, P0, Pf, p, eps, slopes, Lambda):
  """
    Solve the TOV equations for a range of neutron star central pressures. Here we
    integrate the stellar structure equations for pressure values between P0 and Pf,
    calculate the corresponding global stellar properties and store the resulting
    mass, radius and tidal deformability in an array.

    Parameters
        data_matrix: ndarray
            Storage array for the calculated stellar properties.
        P0: float
            Initial neutron star central pressure.
        Pf: float
            Final neutron star central pressure.
        p: ndarray
            Array containing the pressure interpolation points of the EOS.
        eps: ndarray
            Array containing the energy density values corresponding to p.
        slopes: ndarray
            Array containing the derivative of the energy density in each
            interpolation interval.
        Lambda: float
            Shift parameter applied to the EOS above the transition pressure.

    Returns
        data_matrix: ndarray
            Updated storage array containing the central pressure, mass, radius
            and dimensionless tidal deformability for each stellar model.
  """
  if Pf > 1e-2:
    N = int(np.floor( (1e-4 - P0) / (2.5e-6) + (1e-3 - 1e-4)/(2.5e-5) + (1e-2 - 1e-3)/(2.5e-4)))
  elif Pf > 1e-3:
    N = int(np.floor( (1e-4 - P0) / (2.5e-6) + (1e-3 - 1e-4)/(2.5e-5) + (Pf - 1e-3)/(2.5e-4)))
  elif Pf> 1e-4:
    N = int(np.floor((1e-4 - P0)/(2.5e-6) + (Pf - 1e-4)/(2.5e-5)))
  elif Pf > 1e-5:
    N = int(np.floor((Pf - P0)/(2.5e-6)))
  else:
    N = 0

  for i in range(N):
    Radius, u_final = integrator(P0, p, eps, slopes, Lambda)
    
    M = Radius / 2 * (1 - u_final[0])
    y = Radius * u_final[4] / u_final[3]
    C = M / (Radius / Lu * 1e-5) #compactness

    #love number
    k2 = (8 * ((1 - 2 * C)**2) * (C**5) * (2 + 2 * C * (-1 + y) - y)
          ) / ( 5 * ( 2 * C * (6
                               + (C**2) * (26 - 22 * y)
                               - 3 * y
                               + 4 * (C**4) * (1 + y)
                               + 3 * C * (-8 + 5 * y)
                               + (C**3) * (-4 + 6 * y)
                               ) + 3 * ((1 - 2 * C)**2) * (2 + 2 * C * (-1 + y) - y) * np.log(1 - 2 * C)
                               ))
    lamb = (2/3) * k2 * ((Radius / Lu * 1e-5)**5) * (KK**5) / (M**5)
    row_next = np.array([[P0 / rhou, M, Radius / Lu * 1e-5, lamb]])
    data_matrix = np.vstack([data_matrix, row_next])

    if P0 < 1e-4:
      P0 += 2.5e-6
    elif P0 < 1e-3:
      P0 += 2.5e-5
    elif P0 < 1e-2:
      P0 += 2.5e-4
    else:
      P0 += 2.5e-3

  return data_matrix

### Evaluating and filtering candidate EoSs

For each high-density parametrization $(\rho_i,c_{s,i})$, we construct their corresponding complete EoS and evaluate them for all vacuum-energy shifts $\Lambda$.

Each candidate EoS is passed through the TOV solver to generate a stellar sequence over a range of central pressures. The resulting maximum stellar mass is then used as an acceptance criterion for physicality. Only models satisfying $$2.18\,M_\odot < M_{\max} < 2.52\,M_\odot$$ are kept in our data and their TOV results are saved for dataset construction (since physical neutronstars mostly have a mass in this range). This function therefore combines EoS construction, TOV evaluation, physical filtering, and output storage for a single sampled high-density parametrization.

In [11]:
def process_one_j(j, eos_name, eos_base, rho_matrix, cs_matrix, out_dir):
    """
    Generate and evaluate EOS candidates. Subsequently, save accepted TOV results. For the row j
    in the candidate parameters, the function creates an EOS table from eos_base. Also, rho_matrix
    and cs_matrix are used to build the interpolations, solves the TOV equations for various density
    nodes and we determine whether this data is accepted or not. Accepted results are saved in out_dir.

    Parameters
        j: int
            Index deciding which row of our rho_matrix and cs_matrix are applied.
        eos_name: str
            Name of EOS used in file name.
        eos_base: ndarray
            baseline EOS data table
        rho_matrix: ndarray
            Matrix containing density values.
        cs_matrix: ndarray
            Matrix containing speed of sound values.
        out_dir: str
            Directory where we save result files in.

        Returns
        dict
            Dictionary containing amount of values tested, amount of accepted models and list of saved file paths.
    """

    z = j
    print(f"starting j = {j+1}")

    local_total = 0
    local_accepted = 0
    saved = []

    eos_matrix = build(eos_base, rho_matrix, cs_matrix, z, rho_t, rho_fin)

    p = eos_matrix[:, 0]
    eps = eos_matrix[:, 1]
    slopes = np.diff(eps) / np.diff(p)

    #end value eos matrix
    eos_end_p = eos_matrix[-1, 0]

    for Lambda in Lamb_arr:
        local_total += 1

        P0 = 2.5e-5
        if Lambda > 0 and eos_end_p > pc:
            Pf = eos_end_p - Lambda
        else:
            Pf = eos_end_p

        data_matrix = np.empty((0, 4))
        data_matrix = cycle_tov(data_matrix, P0, Pf, p, eps, slopes, Lambda)

        if data_matrix.size == 0:
            continue

        M_max = np.max(data_matrix[:, 1])

        if Lambda == 0:
            temp = 0
        else:
            temp = int(np.floor((abs(Lambda / (Kp * Pu)) ** 0.25) / np.sign(Lambda)))

        if 2.18 < M_max < 2.52:
            local_accepted += 1
            out_path = f"{out_dir}/TOV_{eos_name}_{temp}_{z+1}.csv"
            np.savetxt(out_path, data_matrix)
            saved.append(out_path)

    print(f"done j = {j+1}")
    return {"total": local_total, "accepted": local_accepted, "saved": saved}

### Parallel TOV data generation

To generate the synthetic dataset, the candidate highdensity EoS parametrizations are evaluated in parallel across multiple CPU cores. Each row of $(\rho_i,c_{s,i})$ parameters is processed independently using the pipeline defined above.

For every candidate, the EoS is constructed, TOV sequences are generated for the allowed vacuum energy shifts and lastly physically accepted models are written to disk. After all parallel jobs are finished, the total number of tested and accepted EoSs is reported together with the acceptance rate.

In [12]:
def generate_tovs(eos_name, eos_base, rho_matrix, cs_matrix, out_dir, n_jobs=6):
    """
    Generate TOV solutions for all EOS parameter data. Here, we paralellize over n_jobs amount of CPU cores, for a more efficient
    data generation. We run process_one_j for each row of rho_matrix. When data is accepted, these are saved in out_dir.
    We also print information like acceptance rate to keep track of the data generation.

    Parameters
        eos_name: str
            Name of EOS used in file names.
        eos_base: ndarray
            Baseline EOS data.
        rho_matrix: ndarray
            Matrix containing density values.
        cs_matrix: ndarray
            Matrix containing speed of sound values.
        out_dir: str
            Directory where we save result files in.
        n_jobs : int
            Number of CPU cores parallelized used to generate data (Check how much cores you have has before running!)
    """
    os.makedirs(out_dir, exist_ok=True)

    for f in os.listdir(out_dir):
        os.remove(os.path.join(out_dir, f))

    results = Parallel(n_jobs=n_jobs, backend="loky", verbose=10)(
        delayed(process_one_j)(j, eos_name, eos_base, rho_matrix, cs_matrix, out_dir)
        for j in range(rho_matrix.shape[0]))

    total = sum(r["total"] for r in results)
    accepted = sum(r["accepted"] for r in results)

    print("eos tested:", total)
    print("eos accepted:", accepted)
    print("acceptance rate:", accepted / total if total > 0 else np.nan)

In [13]:
generate_tovs("ap4", ap4, rho_train, cs_train, "TOVs_ap4")

[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done   1 tasks      | elapsed:    9.4s
[Parallel(n_jobs=6)]: Done   6 tasks      | elapsed:   10.3s
[Parallel(n_jobs=6)]: Done  13 tasks      | elapsed:   22.5s
[Parallel(n_jobs=6)]: Done  20 tasks      | elapsed:   29.7s
[Parallel(n_jobs=6)]: Done  29 tasks      | elapsed:   40.6s
[Parallel(n_jobs=6)]: Done  38 tasks      | elapsed:   51.1s
[Parallel(n_jobs=6)]: Done  49 tasks      | elapsed:  1.0min
[Parallel(n_jobs=6)]: Done  60 tasks      | elapsed:  1.3min
[Parallel(n_jobs=6)]: Done  73 tasks      | elapsed:  1.6min
[Parallel(n_jobs=6)]: Done  86 tasks      | elapsed:  1.8min
[Parallel(n_jobs=6)]: Done 101 tasks      | elapsed:  2.1min
[Parallel(n_jobs=6)]: Done 116 tasks      | elapsed:  2.4min
[Parallel(n_jobs=6)]: Done 133 tasks      | elapsed:  2.8min
[Parallel(n_jobs=6)]: Done 150 tasks      | elapsed:  3.1min
[Parallel(n_jobs=6)]: Done 169 tasks      | elapsed:  3.5min
[Parallel(

eos tested: 10000
eos accepted: 1037
acceptance rate: 0.1037


[Parallel(n_jobs=6)]: Done 1000 out of 1000 | elapsed: 20.0min finished


In [14]:
generate_tovs("sly", sly, rho_train, cs_train, "TOVs_sly")

[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done   1 tasks      | elapsed:    6.7s
[Parallel(n_jobs=6)]: Done   6 tasks      | elapsed:    7.5s
[Parallel(n_jobs=6)]: Done  13 tasks      | elapsed:   20.1s
[Parallel(n_jobs=6)]: Done  20 tasks      | elapsed:   27.3s
[Parallel(n_jobs=6)]: Done  29 tasks      | elapsed:   37.1s
[Parallel(n_jobs=6)]: Done  38 tasks      | elapsed:   48.8s
[Parallel(n_jobs=6)]: Done  49 tasks      | elapsed:  1.0min
[Parallel(n_jobs=6)]: Done  60 tasks      | elapsed:  1.3min
[Parallel(n_jobs=6)]: Done  73 tasks      | elapsed:  1.6min
[Parallel(n_jobs=6)]: Done  86 tasks      | elapsed:  1.8min
[Parallel(n_jobs=6)]: Done 101 tasks      | elapsed:  2.1min
[Parallel(n_jobs=6)]: Done 116 tasks      | elapsed:  2.4min
[Parallel(n_jobs=6)]: Done 133 tasks      | elapsed:  2.7min
[Parallel(n_jobs=6)]: Done 150 tasks      | elapsed:  3.1min
[Parallel(n_jobs=6)]: Done 169 tasks      | elapsed:  3.4min
[Parallel(

eos tested: 10000
eos accepted: 1221
acceptance rate: 0.1221


[Parallel(n_jobs=6)]: Done 1000 out of 1000 | elapsed: 20.1min finished


In [15]:
generate_tovs("ap4", ap4, rho_test, cs_test, "TOVsTest_ap4")

[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done   1 tasks      | elapsed:    4.6s
[Parallel(n_jobs=6)]: Done   6 tasks      | elapsed:    7.7s
[Parallel(n_jobs=6)]: Done  13 tasks      | elapsed:   18.2s
[Parallel(n_jobs=6)]: Done  20 tasks      | elapsed:   25.4s
[Parallel(n_jobs=6)]: Done  29 tasks      | elapsed:   36.9s
[Parallel(n_jobs=6)]: Done  38 tasks      | elapsed:   45.6s
[Parallel(n_jobs=6)]: Done  49 tasks      | elapsed:   59.0s
[Parallel(n_jobs=6)]: Done  60 tasks      | elapsed:  1.2min
[Parallel(n_jobs=6)]: Done  73 tasks      | elapsed:  1.5min
[Parallel(n_jobs=6)]: Done  86 tasks      | elapsed:  1.7min


eos tested: 1000
eos accepted: 92
acceptance rate: 0.092


[Parallel(n_jobs=6)]: Done 100 out of 100 | elapsed:  2.0min finished


In [16]:
generate_tovs("sly", sly, rho_test, cs_test, "TOVsTest_sly")

[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done   1 tasks      | elapsed:    4.8s
[Parallel(n_jobs=6)]: Done   6 tasks      | elapsed:    7.3s
[Parallel(n_jobs=6)]: Done  13 tasks      | elapsed:   18.0s
[Parallel(n_jobs=6)]: Done  20 tasks      | elapsed:   25.1s
[Parallel(n_jobs=6)]: Done  29 tasks      | elapsed:   37.0s
[Parallel(n_jobs=6)]: Done  38 tasks      | elapsed:   46.1s
[Parallel(n_jobs=6)]: Done  49 tasks      | elapsed:   59.3s
[Parallel(n_jobs=6)]: Done  60 tasks      | elapsed:  1.2min
[Parallel(n_jobs=6)]: Done  73 tasks      | elapsed:  1.5min
[Parallel(n_jobs=6)]: Done  86 tasks      | elapsed:  1.7min


eos tested: 1000
eos accepted: 115
acceptance rate: 0.115


[Parallel(n_jobs=6)]: Done 100 out of 100 | elapsed:  2.0min finished


### Synthetic mass–radius observations

The accepted TOV solutions define a mass–radius relation for each candidate EoS. To construct synthetic observations, the stable part of the $M$–$R$ curve is interpolated and stellar masses are sampled uniformly across the mass range. The corresponding radii are obtained from the interpolated relation.

Gaussian measurement noise is then added to the mass and radius, $$ M_{\mathrm{obs}} = M + \epsilon_M, \qquad R_{\mathrm{obs}} = R + \epsilon_R,$$

with $\epsilon_M \sim \mathcal{N}(0,\sigma_M^2)$ and $\epsilon_R \sim \mathcal{N}(0,\sigma_R^2)$. The resulting observations are finally
normalised for use as machine-learning inputs.

In [17]:
def data_generator(data_matrix, max_i, sigma_M, sigma_R, TOT):
    """
    Generate radius and mass samples from our EOS curve. Here, we interpolate the mass-radius curve from our data. Masses are sampled and
    radii are computed using the RM curve. Subsequently, Gaussian noise is added to those values with a standard deviation of sigma_M and sigma_R respectively.
    We normalize these values.

    Parameters
        data_matrix: ndarray
            Array containing EOS including mass and radius.
        max_i: int
            Index up to which we use data to interpolate.
        sigma_M: float
            Mass noise standard deviation.
        sigma_R: float
            Radius noise standard deviation
        TOT: int
            Number of samples generated.

    Returns
        R_rand, M_rand: ndarray, ndarray
            Arrays of samples generated.
    """
    M_arr = data_matrix[:, 1]
    R_arr = data_matrix[:, 2]

    M_unique, idx_unique = np.unique(M_arr[:max_i+1], return_index=True)
    R_unique = R_arr[:max_i+1][idx_unique]

    RM_curve = interp1d(M_unique, R_unique, kind='linear', fill_value='extrapolate')
    M_rand = uniform.rvs(loc=M_unique[0], scale=M_unique[-1] - M_unique[0], size=TOT)
    R_rand = RM_curve(M_rand)

    M_rand = norm.rvs(loc=M_rand, scale=sigma_M) / M_norm
    R_rand = norm.rvs(loc=R_rand, scale=sigma_R) / R_norm

    return R_rand, M_rand

### Constructing the machine-learning dataset

The accepted TOV solutions are converted into supervised learning samples by repeatedly drawing noisy mass–radius observations from each stellar sequence. Every sample is paired with the corresponding high-density EoS parameters $(\rho_i,c_{s,i})$ and vacuum energy shift $\Lambda$, producing the final dataset used for training in our  inference models.

Each generated sample therefore has the form $$(\rho_i,c_{s,i},\Lambda)\;\longrightarrow\;(M_1,R_1,\ldots,M_{30},R_{30}), $$ where 30 noisy mass–radius observations are generated from the same underlying EoS.

In [18]:
def generate_mr_dataset(source_dir, dataset_path, rho_matrix, cs_matrix, ns=300, TOT=30):
    """
    This function generates the mass-radius dataset from the data containing the solved TOV equations.
    Here, the function reads the TOV files and generates random mass-radius samples for all files.
    Subsequently, these files are combined with the EOS parameters and then saved.

    Parameters
    source_dir: str
        Directory of saved TOV files.
    dataset_path: str
        Path of final dataset.
    rho_matrix: ndarray
        Matrix containing density values.
    cs_matrix: ndarray
        Matrix containing speed of sound values.
    ns: int
        Number of samples per TOV file.
    TOT: int
        Amount of mass-radius points per sample.

    Returns
        None
    """
    sigma_M = 0.1
    sigma_R = 0.5

    if os.path.exists(dataset_path):
        os.remove(dataset_path)

    listfile = sorted(os.listdir(source_dir))

    for file_idx, filename in enumerate(listfile, start=1):
        print(f"{file_idx}/{len(listfile)} : {filename}")

        parts = filename.split(".")[0].split("_")
        eos_name = parts[1]
        Lambda_temp = int(parts[2])
        z = int(parts[3])

        TOV_matrix = np.loadtxt(os.path.join(source_dir, filename))

        rows_to_write = []

        for _ in range(ns):
            idx_max = np.argmax(TOV_matrix[:, 1])
            R_rand, M_rand = data_generator(TOV_matrix, idx_max, sigma_M, sigma_R, TOT)

            row = np.concatenate([
                np.array([eos_name, Lambda_temp], dtype=object),
                rho_matrix[z - 1, :],
                cs_matrix[z - 1, :],
                M_rand,
                R_rand
            ])

            rows_to_write.append(row)

        rows_to_write = np.array(rows_to_write, dtype=object)

        with open(dataset_path, "ab") as f:
            np.savetxt(f, rows_to_write, fmt="%s")

In [19]:
generate_mr_dataset("TOVs_ap4", "dataset_30points_larger300_ap4.csv", rho_train, cs_train, ns=300, TOT=30)
generate_mr_dataset("TOVs_sly", "dataset_30points_larger300_sly.csv", rho_train, cs_train, ns=300, TOT=30)

1/1037 : TOV_ap4_-120_104.csv
2/1037 : TOV_ap4_-120_109.csv
3/1037 : TOV_ap4_-120_120.csv
4/1037 : TOV_ap4_-120_123.csv
5/1037 : TOV_ap4_-120_124.csv
6/1037 : TOV_ap4_-120_125.csv
7/1037 : TOV_ap4_-120_129.csv
8/1037 : TOV_ap4_-120_132.csv
9/1037 : TOV_ap4_-120_139.csv
10/1037 : TOV_ap4_-120_15.csv
11/1037 : TOV_ap4_-120_150.csv
12/1037 : TOV_ap4_-120_158.csv
13/1037 : TOV_ap4_-120_163.csv
14/1037 : TOV_ap4_-120_174.csv
15/1037 : TOV_ap4_-120_186.csv
16/1037 : TOV_ap4_-120_188.csv
17/1037 : TOV_ap4_-120_206.csv
18/1037 : TOV_ap4_-120_210.csv
19/1037 : TOV_ap4_-120_218.csv
20/1037 : TOV_ap4_-120_236.csv
21/1037 : TOV_ap4_-120_251.csv
22/1037 : TOV_ap4_-120_258.csv
23/1037 : TOV_ap4_-120_259.csv
24/1037 : TOV_ap4_-120_263.csv
25/1037 : TOV_ap4_-120_267.csv
26/1037 : TOV_ap4_-120_268.csv
27/1037 : TOV_ap4_-120_271.csv
28/1037 : TOV_ap4_-120_279.csv
29/1037 : TOV_ap4_-120_287.csv
30/1037 : TOV_ap4_-120_289.csv
31/1037 : TOV_ap4_-120_290.csv
32/1037 : TOV_ap4_-120_315.csv
33/1037 : TOV_ap4_

### k_2 data generation

A similar procedure is followed for the generation of the $M$,$R$,$k_2$ dataset.

In [20]:
def data_generator_k2(data_matrix, max_i, TOT, sigma_M, sigma_R, sigma_k2):
    """
    Generate radius and mass samples from our EOS curve. Here, we interpolate the mass-radius curve from our data. Masses are sampled and
    radii are computed using the RM curve. Subsequently, Gaussian noise is added to those values with a standard deviation of sigma_M and sigma_R respectively.
    We normalize these values.

    Parameters
        data_matrix: ndarray - [P0, M, R, lambda]
            Array containing EOS including mass and radius.
        max_i: int
            Index up to which we use data to interpolate.
        sigma_M: float
            Mass noise standard deviation.
        sigma_R: float
            Radius noise standard deviation.
        TOT: int
            Number of samples generated.
        sigma_k2: float
            k2 noise standard deviation.

    Returns
        R_rand, M_rand, k2_rand: ndarray, ndarray, ndarray
            Arrays of samples generated.
    """
    M_arr = data_matrix[:, 1]
    R_arr = data_matrix[:, 2]
    lamb_arr = data_matrix[:, 3]

    #compute k2 from lambda
    k2_arr = (3.0 / 2.0) * lamb_arr * (M_arr ** 5) / (R_arr ** 5) / (KK ** 5)

    #keep stable branch up to maximum mass
    M_unique, idx_unique = np.unique(M_arr[:max_i+1], return_index=True)
    R_unique = R_arr[:max_i+1][idx_unique]
    k2_unique = k2_arr[:max_i+1][idx_unique]

    RM_curve = interp1d(M_unique, R_unique, kind="linear", fill_value="extrapolate")
    k2M_curve = interp1d(M_unique, k2_unique, kind="linear", fill_value="extrapolate")

    #uniform amss sampling
    M_rand = uniform.rvs(loc=M_unique[0], scale=M_unique[-1] - M_unique[0], size=TOT)
    R_rand = RM_curve(M_rand)
    k2_rand = k2M_curve(M_rand)

    #Gaussian noise injection
    M_rand = norm.rvs(loc=M_rand, scale=sigma_M) / M_norm
    R_rand = norm.rvs(loc=R_rand, scale=sigma_R) / R_norm
    k2_rand = norm.rvs(loc=k2_rand, scale=sigma_k2)

    return R_rand, M_rand, k2_rand

In [21]:
def generate_mrk2_dataset(source_dir, dataset_k2_path, rho_matrix, cs_matrix, ns=100, TOT=30):
    """
    This function generates the mass, radius, k2 dataset from the data containing the solved TOV equations.
    Here, the function reads the TOV files and generates random mass, radius, k2 samples for all files.
    Subsequently, these files are combined with the EOS parameters and then saved.

    Parameters
    source_dir: str
        Directory of saved TOV files.
    dataset_k2_path: str
        Path of final dataset.
    rho_matrix: ndarray
        Matrix containing density values.
    cs_matrix: ndarray
        Matrix containing speed of sound values.
    ns: int
        Number of samples per TOV file.
    TOT: int
        Amount of mass-radius-k2 points per sample.

    Returns
        None
    """
    sigma_k2 = 0.05
    sigma_M = 0.1
    sigma_R = 0.5

    if os.path.exists(dataset_k2_path):
        os.remove(dataset_k2_path)

    listfile = sorted(os.listdir(source_dir))

    for file_idx, filename in enumerate(listfile, start=1):
        num = f"{file_idx}/{len(listfile)}"
        print(f"{num} : {filename}")

        parts = filename.split(".")[0].split("_")
        eos_name = parts[1]
        Lambda_temp = int(parts[2])
        z = int(parts[3])

        TOV_matrix = np.loadtxt(os.path.join(source_dir, filename))

        rows_to_write = []

        idx_max = np.argmax(TOV_matrix[:, 1])

        for i in range(ns):
            R_rand, M_rand, k2_rand = data_generator_k2(TOV_matrix, idx_max, TOT, sigma_M, sigma_R, sigma_k2)

            row = np.concatenate([np.array([eos_name, Lambda_temp], dtype=object), rho_matrix[z - 1, :],
                                  cs_matrix[z-1, :], M_rand, R_rand, k2_rand])

            rows_to_write.append(row)

        rows_to_write = np.array(rows_to_write, dtype=object)

        with open(dataset_k2_path, "ab") as f:
            np.savetxt(f, rows_to_write, fmt="%s")

In [22]:
generate_mrk2_dataset("TOVs_ap4", "datasetk2_30points_larger300_ap4.csv", rho_train, cs_train, ns=300, TOT=30)
generate_mrk2_dataset("TOVs_sly", "datasetk2_30points_larger300_sly.csv", rho_train, cs_train, ns=300, TOT=30)

1/1037 : TOV_ap4_-120_104.csv
2/1037 : TOV_ap4_-120_109.csv
3/1037 : TOV_ap4_-120_120.csv
4/1037 : TOV_ap4_-120_123.csv
5/1037 : TOV_ap4_-120_124.csv
6/1037 : TOV_ap4_-120_125.csv
7/1037 : TOV_ap4_-120_129.csv
8/1037 : TOV_ap4_-120_132.csv
9/1037 : TOV_ap4_-120_139.csv
10/1037 : TOV_ap4_-120_15.csv
11/1037 : TOV_ap4_-120_150.csv
12/1037 : TOV_ap4_-120_158.csv
13/1037 : TOV_ap4_-120_163.csv
14/1037 : TOV_ap4_-120_174.csv
15/1037 : TOV_ap4_-120_186.csv
16/1037 : TOV_ap4_-120_188.csv
17/1037 : TOV_ap4_-120_206.csv
18/1037 : TOV_ap4_-120_210.csv
19/1037 : TOV_ap4_-120_218.csv
20/1037 : TOV_ap4_-120_236.csv
21/1037 : TOV_ap4_-120_251.csv
22/1037 : TOV_ap4_-120_258.csv
23/1037 : TOV_ap4_-120_259.csv
24/1037 : TOV_ap4_-120_263.csv
25/1037 : TOV_ap4_-120_267.csv
26/1037 : TOV_ap4_-120_268.csv
27/1037 : TOV_ap4_-120_271.csv
28/1037 : TOV_ap4_-120_279.csv
29/1037 : TOV_ap4_-120_287.csv
30/1037 : TOV_ap4_-120_289.csv
31/1037 : TOV_ap4_-120_290.csv
32/1037 : TOV_ap4_-120_315.csv
33/1037 : TOV_ap4_

In [23]:
generate_mrk2_dataset("TOVsTest_ap4", "datasetk2_30points_Test_100_ap4.csv", rho_test, cs_test, ns=100, TOT=30)
generate_mrk2_dataset("TOVsTest_sly", "datasetk2_30points_Test_100_sly.csv", rho_test, cs_test, ns=100, TOT=30)

1/92 : TOV_ap4_-120_100.csv
2/92 : TOV_ap4_-120_12.csv
3/92 : TOV_ap4_-120_2.csv
4/92 : TOV_ap4_-120_40.csv
5/92 : TOV_ap4_-120_44.csv
6/92 : TOV_ap4_-120_46.csv
7/92 : TOV_ap4_-120_55.csv
8/92 : TOV_ap4_-120_60.csv
9/92 : TOV_ap4_-120_73.csv
10/92 : TOV_ap4_-120_80.csv
11/92 : TOV_ap4_-120_99.csv
12/92 : TOV_ap4_-150_100.csv
13/92 : TOV_ap4_-150_14.csv
14/92 : TOV_ap4_-150_2.csv
15/92 : TOV_ap4_-150_28.csv
16/92 : TOV_ap4_-150_29.csv
17/92 : TOV_ap4_-150_34.csv
18/92 : TOV_ap4_-150_40.csv
19/92 : TOV_ap4_-150_41.csv
20/92 : TOV_ap4_-150_44.csv
21/92 : TOV_ap4_-150_46.csv
22/92 : TOV_ap4_-150_5.csv
23/92 : TOV_ap4_-150_54.csv
24/92 : TOV_ap4_-150_55.csv
25/92 : TOV_ap4_-150_60.csv
26/92 : TOV_ap4_-150_8.csv
27/92 : TOV_ap4_-150_80.csv
28/92 : TOV_ap4_-150_81.csv
29/92 : TOV_ap4_-150_88.csv
30/92 : TOV_ap4_-150_94.csv
31/92 : TOV_ap4_-150_99.csv
32/92 : TOV_ap4_-194_18.csv
33/92 : TOV_ap4_-194_19.csv
34/92 : TOV_ap4_-194_20.csv
35/92 : TOV_ap4_-194_23.csv
36/92 : TOV_ap4_-194_33.csv
37/